In [ ]:
pip install sentence-transformers transformers torch scikit-learn numpy

In [ ]:
import re
import numpy as np
import torch

from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForSequenceClassification


# ============================================================
# 1. HALLUCINATION DETECTOR
# ============================================================

class HallucinationDetector:

    def __init__(
        self,
        embedding_model="sentence-transformers/all-MiniLM-L6-v2",
        nli_model="cross-encoder/nli-MiniLM2-L6-H768"
    ):
        print("Loading embedding model...")
        self.embedding_model = SentenceTransformer(embedding_model)

        print("Loading NLI model...")
        self.nli_tokenizer = AutoTokenizer.from_pretrained(nli_model)
        self.nli_model = AutoModelForSequenceClassification.from_pretrained(
            nli_model
        )

        self.device = torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        )

        self.nli_model.to(self.device)
        self.nli_model.eval()

        # Labels used by the NLI model
        self.nli_labels = {
            0: "CONTRADICTION",
            1: "ENTAILMENT",
            2: "NEUTRAL"
        }

        print(f"Using device: {self.device}")
        print("Models loaded successfully.\n")

    # ========================================================
    # 2. CLAIM EXTRACTION
    # ========================================================

    def extract_claims(self, text):
        """
        Splits an LLM response into individual claims.

        This is a lightweight baseline claim extractor.
        """

        # Split on ., ! and ?
        sentences = re.split(r'(?<=[.!?])\s+', text.strip())

        claims = []

        for sentence in sentences:
            sentence = sentence.strip()

            if len(sentence) < 10:
                continue

            # Remove bullets/numbers
            sentence = re.sub(
                r'^\s*[-*•\d.)]+\s*',
                '',
                sentence
            )

            if sentence:
                claims.append(sentence)

        return claims

    # ========================================================
    # 3. CREATE EVIDENCE EMBEDDINGS
    # ========================================================

    def encode_evidence(self, evidence):
        """
        Converts evidence documents into embeddings.
        """

        embeddings = self.embedding_model.encode(
            evidence,
            convert_to_tensor=True,
            normalize_embeddings=True
        )

        return embeddings

    # ========================================================
    # 4. RETRIEVE RELEVANT EVIDENCE
    # ========================================================

    def retrieve_evidence(
        self,
        claim,
        evidence,
        evidence_embeddings,
        top_k=3
    ):
        """
        Retrieves the most semantically similar evidence
        passages for a claim.
        """

        claim_embedding = self.embedding_model.encode(
            claim,
            convert_to_tensor=True,
            normalize_embeddings=True
        )

        similarities = util.cos_sim(
            claim_embedding,
            evidence_embeddings
        )[0]

        top_k = min(top_k, len(evidence))

        top_results = torch.topk(
            similarities,
            k=top_k
        )

        results = []

        for score, index in zip(
            top_results.values,
            top_results.indices
        ):
            results.append({
                "text": evidence[index],
                "similarity": float(score)
            })

        return results

    # ========================================================
    # 5. NLI VERIFICATION
    # ========================================================

    def nli_check(self, claim, evidence):
        """
        Determines whether evidence supports,
        contradicts, or is neutral to the claim.
        """

        encoded = self.nli_tokenizer(
            evidence,
            claim,
            truncation=True,
            padding=True,
            return_tensors="pt"
        )

        encoded = {
            key: value.to(self.device)
            for key, value in encoded.items()
        }

        with torch.no_grad():
            output = self.nli_model(**encoded)

        probabilities = torch.softmax(
            output.logits,
            dim=-1
        )[0]

        prediction = torch.argmax(probabilities).item()

        return {
            "label": self.nli_labels[prediction],
            "contradiction": float(probabilities[0]),
            "entailment": float(probabilities[1]),
            "neutral": float(probabilities[2])
        }

    # ========================================================
    # 6. VERIFY A SINGLE CLAIM
    # ========================================================

    def verify_claim(
        self,
        claim,
        evidence,
        evidence_embeddings,
        top_k=3
    ):

        retrieved = self.retrieve_evidence(
            claim,
            evidence,
            evidence_embeddings,
            top_k
        )

        best_result = None

        for item in retrieved:

            nli_result = self.nli_check(
                claim,
                item["text"]
            )

            # ------------------------------------------------
            # Support score
            # ------------------------------------------------

            support_score = (
                0.5 * nli_result["entailment"]
                +
                0.3 * item["similarity"]
                -
                0.7 * nli_result["contradiction"]
            )

            support_score = max(
                0.0,
                min(1.0, support_score)
            )

            hallucination_score = 1.0 - support_score

            result = {
                "claim": claim,
                "evidence": item["text"],
                "similarity": item["similarity"],
                "nli": nli_result,
                "support_score": support_score,
                "hallucination_score": hallucination_score
            }

            if (
                best_result is None
                or result["support_score"]
                > best_result["support_score"]
            ):
                best_result = result

        return best_result

    # ========================================================
    # 7. ANALYZE COMPLETE LLM RESPONSE
    # ========================================================

    def analyze(
        self,
        llm_response,
        evidence,
        top_k=3
    ):

        claims = self.extract_claims(llm_response)

        if len(claims) == 0:
            return {
                "overall_score": 0.0,
                "classification": "NO CLAIMS",
                "claims": []
            }

        print(
            f"Extracted {len(claims)} claims."
        )

        # Encode evidence only once
        evidence_embeddings = self.encode_evidence(
            evidence
        )

        results = []

        for i, claim in enumerate(claims, start=1):

            print(
                f"Checking claim {i}/{len(claims)}..."
            )

            result = self.verify_claim(
                claim,
                evidence,
                evidence_embeddings,
                top_k
            )

            results.append(result)

        # ----------------------------------------------------
        # Overall hallucination score
        # ----------------------------------------------------

        hallucination_scores = [
            result["hallucination_score"]
            for result in results
        ]

        overall_score = float(
            np.mean(hallucination_scores)
        )

        classification = self.classify(
            overall_score
        )

        return {
            "overall_score": overall_score,
            "classification": classification,
            "claims": results
        }

    # ========================================================
    # 8. CLASSIFICATION
    # ========================================================

    def classify(self, score):

        if score < 0.20:
            return "VERY LOW HALLUCINATION"

        elif score < 0.40:
            return "LOW HALLUCINATION"

        elif score < 0.60:
            return "MODERATE HALLUCINATION"

        elif score < 0.80:
            return "HIGH HALLUCINATION"

        else:
            return "VERY HIGH HALLUCINATION"


# ============================================================
# 9. PRINT RESULTS
# ============================================================

def print_results(results):

    print("\n")
    print("=" * 70)
    print("HALLUCINATION DETECTION RESULTS")
    print("=" * 70)

    print(
        f"\nOverall Hallucination Score : "
        f"{results['overall_score']:.4f}"
    )

    print(
        f"Classification               : "
        f"{results['classification']}"
    )

    print("\n")

    for i, result in enumerate(
        results["claims"],
        start=1
    ):

        print("-" * 70)

        print(f"CLAIM {i}")
        print(f"Claim: {result['claim']}")

        print(
            f"\nRetrieved Evidence:"
        )

        print(
            result["evidence"]
        )

        print(
            f"\nSemantic Similarity : "
            f"{result['similarity']:.4f}"
        )

        print(
            f"NLI Prediction      : "
            f"{result['nli']['label']}"
        )

        print(
            f"Entailment Score    : "
            f"{result['nli']['entailment']:.4f}"
        )

        print(
            f"Contradiction Score : "
            f"{result['nli']['contradiction']:.4f}"
        )

        print(
            f"Neutral Score       : "
            f"{result['nli']['neutral']:.4f}"
        )

        print(
            f"Support Score       : "
            f"{result['support_score']:.4f}"
        )

        print(
            f"Hallucination Score : "
            f"{result['hallucination_score']:.4f}"
        )

        if result["nli"]["label"] == "ENTAILMENT":
            print("Verdict             : SUPPORTED")

        elif result["nli"]["label"] == "CONTRADICTION":
            print("Verdict             : HALLUCINATED")

        else:
            print("Verdict             : UNCERTAIN")


# ============================================================
# 10. MAIN PROGRAM
# ============================================================

if __name__ == "__main__":

    # --------------------------------------------------------
    # Knowledge base / trusted evidence
    # --------------------------------------------------------

    evidence = [

        "Virat Kohli is an Indian international cricketer "
        "who has represented India in all three formats of "
        "international cricket.",

        "Virat Kohli was born on 5 November 1988 in Delhi, India.",

        "Cristiano Ronaldo is a Portuguese professional "
        "footballer who has played for several European clubs "
        "and the Portugal national team.",

        "Michael Phelps is an American former competitive swimmer "
        "and is the most decorated Olympian of all time.",

        "The Olympic Games are held every four years.",

        "Python is a high-level programming language commonly "
        "used for machine learning, data science and software development.",

        "Machine learning is a branch of artificial intelligence "
        "that enables computer systems to learn patterns from data.",

        "Deep learning uses neural networks with multiple layers "
        "to learn complex representations from data."
    ]

    # --------------------------------------------------------
    # Example hallucinated LLM response
    # --------------------------------------------------------

    llm_response = """
    Virat Kohli is an Indian international cricketer.
    He was born on 5 November 1988 in Delhi.
    Virat Kohli was born in Mumbai in 1990.
    Cristiano Ronaldo is a Portuguese footballer.
    Michael Phelps is the most decorated Olympian of all time.
    The Olympic Games are held every two years.
    """

    # --------------------------------------------------------
    # Initialize detector
    # --------------------------------------------------------

    detector = HallucinationDetector()

    # --------------------------------------------------------
    # Analyze response
    # --------------------------------------------------------

    results = detector.analyze(
        llm_response,
        evidence,
        top_k=3
    )

    # --------------------------------------------------------
    # Display results
    # --------------------------------------------------------

    print_results(results)

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (15 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  328MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Using device: cpu
Models loaded successfully.

Extracted 6 claims.
Checking claim 1/6...
Checking claim 2/6...
Checking claim 3/6...
Checking claim 4/6...
Checking claim 5/6...
Checking claim 6/6...


HALLUCINATION DETECTION RESULTS

Overall Hallucination Score : 0.4462
Classification               : MODERATE HALLUCINATION


----------------------------------------------------------------------
CLAIM 1
Claim: Virat Kohli is an Indian international cricketer.

Retrieved Evidence:
Virat Kohli is an Indian international cricketer who has represented India in all three formats of international cricket.

Semantic Similarity : 0.9223
NLI Prediction      : ENTAILMENT
Entailment Score    : 0.9920
Contradiction Score : 0.0008
Neutral Score       : 0.0072
Support Score       : 0.7722
Hallucination Score : 0.2278
Verdict             : SUPPORTED
----------------------------------------------------------------------
CLAIM 2
Claim: He was born on 5 November 1988 in Delhi.

Retrieved Evidence:
Virat 